In [1]:
import pandas as pd
from pathlib import Path

In [2]:
#Entrando no caminho dos arquivos
DATA_PATH = Path("../data/raw")

files = list(DATA_PATH.glob("*.csv"))

for file in files:
    print(file.name)

olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_orders_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


In [3]:
# Carregando os datasets em um dicionário
# CEP é lido como string: como inteiro os zeros à esquerda se perdem (ex.: 01046 -> 1046)
zip_dtypes = {
    "customer_zip_code_prefix": str,
    "seller_zip_code_prefix": str,
    "geolocation_zip_code_prefix": str,
}

datasets = {}

for file in files:
    name = file.stem
    datasets[name] = pd.read_csv(file, dtype=zip_dtypes)

print(f"{len(datasets)} datasets carregados.")

9 datasets carregados.


In [4]:
# Printando a forma de cada dataset
for name, df in datasets.items():
    print(f"{name}: {df.shape}")

olist_customers_dataset: (99441, 5)
olist_geolocation_dataset: (1000163, 5)
olist_orders_dataset: (99441, 8)
olist_order_items_dataset: (112650, 7)
olist_order_payments_dataset: (103886, 5)
olist_order_reviews_dataset: (99224, 7)
olist_products_dataset: (32951, 9)
olist_sellers_dataset: (3095, 4)
product_category_name_translation: (71, 2)


In [5]:
# Carregando dataframes
geolocation_df = pd.DataFrame(datasets["olist_geolocation_dataset"])
customers_df = pd.DataFrame(datasets["olist_customers_dataset"])
sellers_df = pd.DataFrame(datasets["olist_sellers_dataset"])
geolocation_df.shape

(1000163, 5)

In [6]:
geolocation_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  str    
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  str    
 4   geolocation_state            1000163 non-null  str    
dtypes: float64(2), str(3)
memory usage: 38.2 MB


In [7]:
#Avaliando se existe valores nulos
geolocation_df.isna().sum()

geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

In [8]:
#Avaliando se existe dados duplicados
n_dup = geolocation_df.duplicated().sum()
print(f"Linhas totalmente duplicadas: {n_dup} ({n_dup / len(geolocation_df) * 100:.2f}%)")

Linhas totalmente duplicadas: 261831 (26.18%)


In [9]:
# Tabela sem chave primária: vários pontos por prefixo de CEP
points_per_zip = geolocation_df.groupby('geolocation_zip_code_prefix').size()

print(f"Prefixos de CEP distintos: {len(points_per_zip)}")
points_per_zip.describe()

Prefixos de CEP distintos: 19015


count    19015.000000
mean        52.598633
std         72.057907
min          1.000000
25%         10.000000
50%         29.000000
75%         66.500000
max       1146.000000
dtype: float64

In [10]:
geolocation_df[["geolocation_lat", "geolocation_lng"]].describe()

,geolocation_lat,geolocation_lng
count,1.000163e+06,1.000163e+06
mean,-2.117615e+01,-4.639054e+01
std,5.715866e+00,4.269748e+00
min,-3.660537e+01,-1.014668e+02
25%,-2.360355e+01,-4.857317e+01
50%,-2.291938e+01,-4.663788e+01
75%,-1.997962e+01,-4.376771e+01
max,4.506593e+01,1.211054e+02


In [11]:
# Coordenadas fora do Brasil (caixa aproximada do território)
LAT_MIN, LAT_MAX = -33.8, 5.3
LNG_MIN, LNG_MAX = -74.0, -34.7

outside = (
    ~geolocation_df['geolocation_lat'].between(LAT_MIN, LAT_MAX) |
    ~geolocation_df['geolocation_lng'].between(LNG_MIN, LNG_MAX)
)
print(f"Pontos fora do Brasil: {outside.sum()}")
geolocation_df.loc[outside].head(10)

Pontos fora do Brasil: 42


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
387565,18243,28.008978,-15.536867,bom retiro da esperanca,SP
513631,28165,41.614052,-8.411675,vila nova de campos,RJ
513643,28155,-34.586422,-58.732101,santa maria,RJ
513754,28155,42.439286,13.820214,santa maria,RJ
514429,28333,38.381672,-6.328200,raposo,RJ
516682,28595,43.684961,-7.411080,portela,RJ
538512,29654,29.409252,-98.484121,santo antônio do canaã,ES
538557,29654,21.657547,-101.466766,santo antonio do canaa,ES
585242,35179,25.995203,-98.078544,santana do paraíso,MG
585260,35179,25.995245,-98.078533,santana do paraiso,MG


In [12]:
UFS = {"AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO", "MA", "MT", "MS", "MG", "PA",
       "PB", "PR", "PE", "PI", "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO"}

print(f"Estados inválidos: {(~geolocation_df['geolocation_state'].isin(UFS)).sum()}")
print(f"CEPs com tamanho diferente de 5: {(geolocation_df['geolocation_zip_code_prefix'].str.len() != 5).sum()}")

Estados inválidos: 0


CEPs com tamanho diferente de 5: 0


In [13]:
# Cidades: acentos, codificação de URL, pontos e espaços
cities = geolocation_df['geolocation_city']

print(f"Cidades distintas (grafias): {cities.nunique()}")
print(f"Com codificação de URL (%26apos%3b): {cities.str.contains('%').sum()}")
print(f"Com espaços nas pontas: {(cities != cities.str.strip()).sum()}")
print(f"Começando com ponto: {cities.str.startswith('.').sum()}")
print(f"Variações de São Paulo:\n{cities[cities.str.contains(r'^s.?o ?paulo$')].value_counts()}")

Cidades distintas (grafias): 8011


Com codificação de URL (%26apos%3b): 2


Com espaços nas pontas: 1


Começando com ponto: 1


Variações de São Paulo:
geolocation_city
sao paulo    135800
são paulo     24918
sãopaulo          2
Name: count, dtype: int64


In [14]:
# Funções para padronizar a cidade (mesmo padrão de customers e sellers: sem acento e minúscula)
import html
import unicodedata
from urllib.parse import unquote

def normalize_city(text):
    text = html.unescape(unquote(str(text)))   # d%26apos%3balho -> d'alho
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode()
    return text.lower().strip().lstrip(".").strip()

# Correções pontuais de grafia
city_fix = {"saopaulo": "sao paulo"}

In [15]:
#Fazendo copia do dataframe para não alterar o original
geolocation_clean = geolocation_df.copy()

In [16]:
# Padronizando cidade (aplicado nas grafias distintas e mapeado, para ser rápido) e estado
city_map = {c: normalize_city(c) for c in geolocation_clean['geolocation_city'].unique()}
city_map = {original: city_fix.get(new, new) for original, new in city_map.items()}

geolocation_clean['geolocation_city'] = geolocation_clean['geolocation_city'].map(city_map)
geolocation_clean['geolocation_state'] = geolocation_clean['geolocation_state'].str.strip().str.upper()
print(f"Cidades distintas depois da padronização: {geolocation_clean['geolocation_city'].nunique()}")

Cidades distintas depois da padronização: 5961


In [17]:
# Removendo pontos com coordenadas fora do Brasil
outside_clean = (
    ~geolocation_clean['geolocation_lat'].between(LAT_MIN, LAT_MAX) |
    ~geolocation_clean['geolocation_lng'].between(LNG_MIN, LNG_MAX)
)
print(f"Pontos removidos por estarem fora do Brasil: {outside_clean.sum()}")
geolocation_clean = geolocation_clean.loc[~outside_clean]

Pontos removidos por estarem fora do Brasil: 42


In [18]:
# Removendo linhas duplicadas (a padronização da cidade faz aparecer mais duplicadas)
before = len(geolocation_clean)
geolocation_clean = geolocation_clean.drop_duplicates().reset_index(drop=True)
print(f"Linhas duplicadas removidas: {before - len(geolocation_clean)}")
print(f"Linhas: {len(geolocation_clean)} (original: {len(geolocation_df)})")

Linhas duplicadas removidas: 279660
Linhas: 720461 (original: 1000163)


In [19]:
# O mesmo CEP aparece em mais de um estado?
states_per_zip = geolocation_clean.groupby('geolocation_zip_code_prefix')['geolocation_state'].nunique()
print(f"CEPs com mais de um estado: {(states_per_zip > 1).sum()}")

print(f"Prefixos de CEP antes: {geolocation_df['geolocation_zip_code_prefix'].nunique()} | depois: {geolocation_clean['geolocation_zip_code_prefix'].nunique()}")

CEPs com mais de um estado: 8
Prefixos de CEP antes: 19015 | depois: 19010


In [20]:
# Tabela de geolocalização por CEP: 1 linha por prefixo (mediana das coordenadas, cidade e estado mais frequentes)
# Necessária para juntar com customers e sellers sem multiplicar linhas
geolocation_zip = (
    geolocation_clean
    .groupby('geolocation_zip_code_prefix')
    .agg(
        geolocation_lat=('geolocation_lat', 'median'),
        geolocation_lng=('geolocation_lng', 'median'),
        geolocation_city=('geolocation_city', lambda s: s.mode().iat[0]),
        geolocation_state=('geolocation_state', lambda s: s.mode().iat[0]),
        n_points=('geolocation_lat', 'size'),
    )
    .reset_index()
)
geolocation_zip.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,n_points
0,01001,-23.550107,-46.634027,sao paulo,SP,10
1,01002,-23.548228,-46.635247,sao paulo,SP,6
2,01003,-23.548976,-46.635318,sao paulo,SP,10
3,01004,-23.549550,-46.634771,sao paulo,SP,14
4,01005,-23.549690,-46.636532,sao paulo,SP,11


In [21]:
# Validações pós-tratamento
print(f"Linhas duplicadas: {geolocation_clean.duplicated().sum()}")
print(f"Valores nulos: {geolocation_clean.isna().sum().sum()}")
print(f"Pontos fora do Brasil: {(~geolocation_clean['geolocation_lat'].between(LAT_MIN, LAT_MAX) | ~geolocation_clean['geolocation_lng'].between(LNG_MIN, LNG_MAX)).sum()}")
print(f"Estados inválidos: {(~geolocation_clean['geolocation_state'].isin(UFS)).sum()}")
print(f"Cidades com codificação de URL: {geolocation_clean['geolocation_city'].str.contains('%').sum()}")
print(f"Linhas na tabela por CEP: {len(geolocation_zip)} | CEPs únicos: {geolocation_zip['geolocation_zip_code_prefix'].is_unique}")

miss_customers = (~customers_df['customer_zip_code_prefix'].isin(geolocation_zip['geolocation_zip_code_prefix'])).sum()
miss_sellers = (~sellers_df['seller_zip_code_prefix'].isin(geolocation_zip['geolocation_zip_code_prefix'])).sum()
print(f"Clientes com CEP sem geolocation: {miss_customers} | Vendedores: {miss_sellers}")

assert not geolocation_clean.duplicated().any()
assert geolocation_zip['geolocation_zip_code_prefix'].is_unique

Linhas duplicadas: 0
Valores nulos: 0
Pontos fora do Brasil: 0
Estados inválidos: 0


Cidades com codificação de URL: 0
Linhas na tabela por CEP: 19010 | CEPs únicos: True
Clientes com CEP sem geolocation: 279 | Vendedores: 7


In [22]:
# Salvando dataframes tratados em pasta processed
geolocation_clean.to_csv('../data/processed/olist_geolocation_dataset_clean.csv', index=False)
geolocation_zip.to_csv('../data/processed/olist_geolocation_zip_clean.csv', index=False)

## 📌 Conclusão da etapa — Geolocation

### 🔎 O que foi encontrado?

A tabela `geolocation` possui **1.000.163 registros e 5 colunas** e não tem chave primária: cada prefixo de CEP tem vários pontos (mediana de 29 pontos por CEP, 19.015 CEPs).

Durante o profiling, foram identificados:

- Nenhum valor nulo, estado inválido ou CEP com tamanho diferente de 5;
- **261.831 linhas totalmente duplicadas (26,18%)**;
- **42 pontos com coordenadas fora do Brasil** (ex.: latitude 45 / longitude 121 para CEPs de SP e RJ);
- **8.011 grafias de cidade**, com acento (`são paulo` × `sao paulo` × `sãopaulo`), codificação de URL (`d%26apos%3balho`), espaço nas pontas e ponto no início.

### 🛠️ O que foi feito?

- Cidade padronizada no mesmo padrão de `customers` e `sellers` (minúscula, sem acento): **8.011 → 5.961 grafias**;
- Removidos os **42 pontos fora do Brasil**;
- Removidas as linhas duplicadas, incluindo as que apareceram após a padronização da cidade: **279.660 linhas** (1.000.163 → **720.461**);
- Criada uma tabela auxiliar **por CEP** (1 linha por prefixo: mediana de latitude e longitude, cidade e estado mais frequentes e `n_points`), necessária para juntar com `customers` e `sellers` sem multiplicar linhas (sem agregar, o join com `customers` gera mais de 15 milhões de linhas).

### ✅ Como foi validado?

- Sem linhas duplicadas, sem nulos e sem pontos fora do Brasil;
- Sem estados inválidos e sem cidades com codificação de URL;
- A tabela por CEP tem **19.010 CEPs únicos**.

### ⚠️ Pontos de atenção

- **5 CEPs** (19.015 → 19.010) só tinham pontos fora do Brasil e deixaram de existir, então clientes sem CEP em `geolocation` passaram de 278 para **279** (vendedores continuam 7);
- **8 CEPs** aparecem em mais de um estado: na tabela por CEP prevalece o estado mais frequente.

### 📁 Resultado

Os datasets tratados foram salvos em:

- `data/processed/olist_geolocation_dataset_clean.csv` (pontos, 720.461 linhas)
- `data/processed/olist_geolocation_zip_clean.csv` (1 linha por CEP, 19.010 linhas)